# Model Analysis

## Logistic Regression

This model aims to solve a binary classification problem predicting `is_approved`, and having a large dataset, Logistic Regression is chosen to solve this problem.

### Data

The model is trained on `data/features/X_train_prep.csv`, built by `scripts/feature_engineering.py`:
- One-to-four family homes with applicants who are people (multifamily and "not applicable" race rows removed); withdrawn, incomplete and purchased applications removed.
- **No protected attributes** (race, sex, ethnicity) are model inputs. They're saved separately for the fairness audit. The script also writes `X_*_prep_with_protected.csv` for comparing a model trained with them.
- Lenders are target-encoded, county is one-hot encoded, and income / loan amount are capped and logged using training-split values only.

### Hyperparameters

Logistic Regression Model
- `class_weight='balanced'`: Handles the severe class imbalance by penalizing mistakes on the minority class more heavily.

GridSearchCV param-grid(Grid Search for optimal hyperparameters)
- `penalty=['l2']` & `solver=['lbfgs']`: grid restricted to L2 regularization with `lbfgs` solver for optimal speed and convergence on massive dataset.
- `C=[0.1, 1, 10, 100]`: Large dataset -> risk of overfitting is low. Focused search on moderate to weak regularization strengths.
- **Grid Search Settings**: `cv=5` 5-fold cross-validation , `f1-score` to properly evaluate imbalanced classes. `n_jobs=-1` for fast parallel processing.

In [1]:
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# Load model
model_path = '../data/models/logistic_regression_model.joblib'
model = joblib.load(model_path)

# Load test data
X_test = pd.read_csv('../data/features/X_test_prep.csv', index_col='row_id')
y_test = pd.read_csv('../data/features/y_test.csv', index_col='row_id')['is_approved']

In [2]:
# Generate predictions and probabilities
y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

# Calculate evaluation metrics
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)

print("--- Logistic Regression Evaluation Metrics ---")
print(f"Accuracy:  {accuracy:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"AUC-ROC:   {roc_auc:.4f}")

--- Logistic Regression Evaluation Metrics ---
Accuracy:  0.7344
F1 Score:  0.8088
AUC-ROC:   0.7921
